# Phase 3 Decomposition — Google Colab (T4)

Runs the two bias-correction methods (symmetric loss, subset-conditional reweighting) and their combination on the mixed HH-RLHF pool, at all three seeds.

**Before running:** `Runtime → Change runtime type → T4 GPU`.

This notebook:
1. Verifies the GPU
2. Installs **pinned** deps (transformers 5.9.0, trl 1.5.0)
3. Clones the repo (`phase3-decomposition` branch) via a GitHub token in Colab Secrets
4. Mounts Drive (results persist there; the run is resumable across sessions)
5. Reconstructs subset tags + computes reweighting weights (with the thin-cell check)
6. Trains + evaluates all 9 arms (symloss / reweight / combined × seeds 42,0,1), syncing to Drive after each
7. Prints the results table vs the pre-registered decision table


## 1. Check GPU


In [ ]:
import torch
if not torch.cuda.is_available():
    raise RuntimeError('No GPU — Runtime → Change runtime type → T4 GPU')
print('GPU:', torch.cuda.get_device_name(0))
print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')


## 2. Install dependencies (pinned to match the local run)


In [ ]:
%pip install -q \
    'transformers==5.9.0' \
    'trl==1.5.0' \
    'datasets>=2.14.0' \
    'accelerate>=0.27.0' \
    'scipy' 'numpy'


## 3. Clone the repo (phase3-decomposition branch)

Private repo. Add your GitHub token to Colab Secrets first:
- Left sidebar → 🔑 **Secrets** → **Add new secret**
- Name `GITHUB_TOKEN`, value a token with `repo` scope; toggle **Notebook access** on


In [ ]:
import os, subprocess
REPO_DIR = '/content/rlhf-bias-decomp'
BRANCH = 'phase3-decomposition'
try:
    from google.colab import userdata
    token = userdata.get('GITHUB_TOKEN')
    print('Token loaded from Colab Secrets.')
except Exception:
    import getpass
    token = getpass.getpass('Paste your GitHub token (hidden): ')
REPO_URL = f'https://{token}@github.com/moisheu/rlhf-bias-decomp.git'
if os.path.exists(REPO_DIR):
    subprocess.run(['git','-C',REPO_DIR,'fetch','origin',BRANCH], check=True)
    subprocess.run(['git','-C',REPO_DIR,'checkout',BRANCH], check=True)
    subprocess.run(['git','-C',REPO_DIR,'pull','origin',BRANCH], check=True)
else:
    subprocess.run(['git','clone','--branch',BRANCH,REPO_URL,REPO_DIR], check=True)
os.chdir(REPO_DIR)
print('cwd:', os.getcwd())
print(subprocess.run(['git','log','--oneline','-1'],capture_output=True,text=True).stdout)


## 4. Mount Google Drive (persistent, resumable)


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
DRIVE = '/content/drive/MyDrive/rlhf-bias-decomp/phase3'
os.makedirs(DRIVE, exist_ok=True)
print('Drive target:', DRIVE)


## 5. Subset tags + reweighting weights

Deterministic (seed=42 pool) — recomputed here so Colab uses the exact same weights as the local validation. Watch for: tag coverage ~100%, and weighted chosen-longer ≈ 50%.


In [ ]:
!python -m experiments.decomposition.build_subset_tags
print('\n' + '='*60 + '\n')
!python -m experiments.decomposition.compute_weights


## 6. Train + evaluate all 9 arms

Order is first-seeds-first (each method's seed-42 result lands first). Full batch-16 on the T4. Results sync to Drive after every arm, and the loop skips any arm already present in the Drive results file — so if the session disconnects, just re-run this cell to resume.


In [ ]:
import json, shutil, subprocess, sys
CORR = 'results/phase3_length_corr.json'
DRIVE_CORR = f'{DRIVE}/phase3_length_corr.json'
SAVE_CKPT_TO_DRIVE = True   # ~260 MB/arm (~2.3 GB total); set False to skip
os.makedirs('results', exist_ok=True)
# resume: pull any prior results back from Drive
if os.path.exists(DRIVE_CORR):
    shutil.copy(DRIVE_CORR, CORR); print('Restored prior results from Drive.')
def done():
    return {r['label'] for r in json.load(open(CORR))} if os.path.exists(CORR) else set()
ARMS = [('symloss',42),('reweight',42),('combined',42),
        ('symloss',0),('reweight',0),('combined',0),
        ('symloss',1),('reweight',1),('combined',1)]
for method, seed in ARMS:
    tag = f'{method}_seed{seed}'
    if tag in done():
        print(f'SKIP {tag} (already done)'); continue
    outdir = f'results/reward_model_{tag}'
    shutil.rmtree(outdir, ignore_errors=True)
    env = {**os.environ, 'METHOD':method, 'TRAIN_SEED':str(seed), 'TRAIN_BATCH':'16'}
    print(f'\n===== TRAIN {tag} =====', flush=True)
    if subprocess.run([sys.executable,'-u','-m','experiments.decomposition.train_phase3'],env=env).returncode:
        raise RuntimeError(f'train failed: {tag}')
    print(f'===== EVAL {tag} =====', flush=True)
    if subprocess.run([sys.executable,'-u','-m','experiments.decomposition.eval_length_correlation',
                       '--model-dir',outdir,'--label',tag,'--out',CORR],env=env).returncode:
        raise RuntimeError(f'eval failed: {tag}')
    shutil.copy(CORR, DRIVE_CORR)
    if SAVE_CKPT_TO_DRIVE:
        dst = f'{DRIVE}/checkpoints/{tag}'; os.makedirs(dst, exist_ok=True)
        for f in ('model.safetensors','config.json','tokenizer.json','tokenizer_config.json','vocab.txt'):
            p = os.path.join(outdir,f)
            if os.path.exists(p): shutil.copy(p, dst)
    print(f'Synced {tag} to Drive.')
print('\nAll arms complete.')


## 7. Results vs the pre-registered decision table


In [ ]:
!python -m experiments.decomposition.summarize_phase3
